## 1.自定义工具
- 工具就是函数+装饰器

### 1.1.定义参数比较简单的工具

In [1]:
#定义一个查询天气的tool
from langchain_core.tools import tool

@tool
def getWeather(location:str,unit:str="Celsius",include_forecast:bool=False)->str:
    """
    查询天气的tool
    Args:
        location: 查询天气的地点
        unit: 摄氏度或者华氏度
        include_forecast: 是否包含天气预报
    """
    #示例代码
    temp=22 if unit=="Celsius" else 71 #如果单位是摄氏度，则温度为22度，否则为71度
    result=f"{location}的天气是晴天，温度为{temp}degrees {unit[0].upper()}。" #温度单位取第一个字符，并转大写
    if include_forecast:
        result+=f"{location}的天气预报是：多云，最高温度为{temp+5}degrees {unit[0].upper()}，最低温度为{temp-5}degrees {unit[0].upper()}。"
    return result

### 1.2. 定义复杂参数的工具
- 使用Pydantic Model描述参数
- 用一个类来封装函数的参数描述

In [3]:
from pydantic import BaseModel,Field    # Field作用是添加描述，BaseModel作用是定义数据结构
from typing import Literal  # Literal 枚举类型

class WeatherInput(BaseModel):
    location:str=Field(description="城市名称")
    unit:Literal["Celsius","Fahrenheit"]=Field(
        default="Celsius",  #默认值
        description="温度单位"
    )
    include_forecast:bool=Field(
        default=True,
        description="是否包含预报"
    )

@tool(args_schema=WeatherInput)
def get_weather(location:str,unit:str="Celsius",include_forecast:bool=True):
    """获取天气信息
    """
    temp=22 if unit=="Celsius" else 71 #如果单位是摄氏度，则温度为22度，否则为71度
    result=f"{location}的天气是晴天，温度为{temp}degrees {unit[0].upper()}。" #温度单位取第一个字符，并转大写
    if include_forecast:
        result+=f"{location}的天气预报是：多云，最高温度为{temp+5}degrees {unit[0].upper()}，最低温度为{temp-5}degrees {unit[0].upper()}。"
    return result
    
        

### 1.3.测试

In [5]:
#调用Agent和工具
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent=create_agent(
    "deepseek-chat",
    tools=[get_weather]
)

response=agent.invoke({
    "messages":[
        HumanMessage(content="江门今天天气如何?")
    ]
})

print(response)

{'messages': [HumanMessage(content='江门今天天气如何?', additional_kwargs={}, response_metadata={}, id='6189c5bb-605c-400e-a127-97cbe845a1c1'), AIMessage(content='好的，我来查一下江门今天的天气情况。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 342, 'total_tokens': 434, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 86}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '86246a09-71de-408d-861b-37f772a3cfbf', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e4ae7-7d86-7c93-9b95-89f641f95c8b-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '江门', 'unit': 'Celsius', 'include_forecast': True}, 'id': 'call_00_44W1JaOIvfNVtuOj5wXi3341', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3

In [7]:
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

江门今天天气如何?
================================== Ai Message ==================================

好的，我来查一下江门今天的天气情况。
Tool Calls:
  get_weather (call_00_44W1JaOIvfNVtuOj5wXi3341)
 Call ID: call_00_44W1JaOIvfNVtuOj5wXi3341
  Args:
    location: 江门
    unit: Celsius
    include_forecast: True
================================= Tool Message =================================
Name: get_weather

江门的天气是晴天，温度为22degrees C。江门的天气预报是：多云，最高温度为27degrees C，最低温度为17degrees C。
================================== Ai Message ==================================

以下是江门今天的天气情况：

🌤 **江门今日天气**

- **当前天气：** 晴天 ☀️
- **当前温度：** 22°C
- **今日预报：** 多云
- **最高温度：** 27°C
- **最低温度：** 17°C

整体来说天气不错，白天温暖舒适，适合外出活动，不过早晚温差较大（最低17°C），建议备一件薄外套哦！


## 2.使用预定义工具
演示web搜索工具：tavily

In [8]:
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os
load_dotenv()

api_key=os.getenv("TAVILY_API_KEY")
#使用langchain里预定义的工具
web_search_tool=TavilySearch(
    max_results=2,
    topic="general",
)



In [7]:
#可是先打印看看输出的格式是什么样子
web_search_tool.invoke("奶块是什么？")

{'query': '奶块是什么？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://baike.baidu.com/item/%E5%A5%B6%E5%9D%97/20483071',
   'title': '奶块_百度百科',
   'content': '《奶块》是国内自研大型沙盒社交手游，开放世界，万人同服，共同创造个性世界。在奶块的世界中，不仅可以制造各种有意思的玩具，更能随心所欲建造一个自己的别墅家园。或者邀请',
   'score': 0.8827621,
   'raw_content': None},
  {'url': 'https://zh.moegirl.org.cn/%E5%A5%B6%E5%9D%97',
   'title': '奶块 - 萌娘百科',
   'content': '《奶块》（英语：NetCraft/英语：MilkCraft/英语：LusoriCraft）是一款由广州虎牙信息科技有限公司开发、广州华多网络科技有限公司发行的故事背景基于北欧',
   'score': 0.8358192,
   'raw_content': None}],
 'response_time': 0.0,
 'request_id': '22d44f61-42ed-4dcd-929a-1f40b84521a2'}

In [9]:
#创建Agent
clint = create_agent(
    model="deepseek-chat",
    tools=[web_search_tool]
)

In [10]:
#调用Agent
response=clint.invoke({
    "messages":[HumanMessage(content="你认识五邑大学梁颖杰吗？")]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

你认识五邑大学梁颖杰吗？
================================== Ai Message ==================================

让我来搜索一下相关信息。
Tool Calls:
  tavily_search (call_00_Es4XB4ppF8MEqIUb34CH5916)
 Call ID: call_00_Es4XB4ppF8MEqIUb34CH5916
  Args:
    query: 五邑大学 梁颖杰
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "五邑大学 梁颖杰", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.wyu.edu.cn/xxgk2/xrld.htm", "title": "现任领导 - 五邑大学", "content": "负责本科教学、实验室与设备管理、学报、后勤、招投标工作；联系轨道交通学院、应用物理与材料学院、纺织材料", "score": 0.24516675, "raw_content": null}, {"url": "https://www.wyu.edu.cn/xuxgk/xrld.htm", "title": "现任领导 - 五邑大学", "content": "负责本科教学、招生、实验室与设备管理、保卫、继续教育工作。分管教务处、实验室与设备管理处、大型仪器共享中心、保卫工作部（武装部、保卫处、退役军人服务中心）、继续", "score": 0.23524933, "raw_content": null}], "response_time": 0.76, "request_id": "f7034811-c0b1-4e02-bff7-8

## 3.优化：预定义工具+自定义
- 自定义‘预定义工具’，让它结构化输出    
1.自定义可以：可以提出不必要的参数，减少token浪费   
2.结构化输出：让结果包含数据源

In [4]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from pydantic import BaseModel,Field
from langchain.messages import HumanMessage


#使用tavily作为网页搜索工具
tavily=TavilySearch(
    max_results=2,
    topic="general",
)


#自定义工具
@tool
def search_tool(query:str):
    """"
    帮助用户查询网页信息
    """
    return tavily.invoke(query)



#结构化Agent回答内容：网页信息+回答信息
class webInfo(BaseModel):
    title:str=Field(description="网页标题")
    url:str=Field(description="网页链接")

class webresult(BaseModel):
    answer:str=Field(description="最终的回答")
    web_info:list[webInfo]=Field(description="网页信息")


#创建Agent
clint=create_agent(
    "deepseek-chat",
    tools=[search_tool],
    response_format=webresult
)

In [ ]:
#调用Agent
response=clint.invoke({
    "messages":[
        HumanMessage(content="五邑大学北门准备什么时候重建？")
    ]
})

In [7]:
print(response['structured_response'])

answer='根据目前公开的信息，五邑大学北门将进行焕新工程（即重建/改造），但官方尚未公布具体的动工（重建）时间。\n\n目前了解到的进展如下：\n1. **设计方案投票阶段**：2026年4月21日，五邑大学校友会发布消息，学校北门将启动焕新工程，并公布了3套改造设计方案。\n2. **投票时间**：即日起至**2026年4月26日24:00**，面向全体师生、校友及社会公众进行投票。\n3. **当前状态**：仍处于方案征集和投票阶段，具体何时动工重建尚未公布。\n\n建议您关注**五邑大学官网**（www.wyu.edu.cn）或**五邑大学校友会**的官方通知，以获取北门重建的最新动态和具体时间安排。' web_info=[webInfo(title='五邑大学北门焕新计划方案以及投票 - 江门本地宝', url='http://jm.bendibao.com/news/2026423/19467.shtm'), webInfo(title='方案曝光！五邑大学北门，即将“大变样”！ - 网易', url='https://www.163.com/dy/article/KR5E7JA40514DG5K.html'), webInfo(title='3套改造设计方案出炉！ 五邑大学北门将“焕新升级”', url='https://m.mp.oeeee.com/a/BAAFRD0000202604221561924.html')]
